In [3]:
import xarray as xr
import numpy as np
import os

In [4]:

# Load ORCESTRA Level-4 dataset
# Downloads from IPFS on first run, then loads locally on subsequent runs.
# If the cached file is missing required variables, it re-downloads automatically.

LOCAL_PATH = "/g/data/k10/zr7147/raw/orcestra_level4.nc"
REQUIRED_VARS = ['omega', 'p_mean', 'circle_time']

def _cache_is_valid(path, required_vars):
    if not os.path.exists(path):
        return False, "file not found"
    try:
        _ds = xr.open_dataset(path)
        missing = [v for v in required_vars if v not in _ds]
        _ds.close()
        if missing:
            return False, f"missing variables: {missing}"
        return True, "ok"
    except Exception as e:
        return False, f"cannot open: {e}"

valid, reason = _cache_is_valid(LOCAL_PATH, REQUIRED_VARS)

if valid:
    print(f"Loading from local cache: {LOCAL_PATH}")
    ds = xr.open_dataset(LOCAL_PATH)
    print(f"Loaded successfully. Variables: {list(ds.data_vars)}")
else:
    print(f"Cache invalid ({reason}) — re-downloading from IPFS...")
    if os.path.exists(LOCAL_PATH):
        os.remove(LOCAL_PATH)
    os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)
    os.environ["IPFS_GATEWAY"] = "https://ipfs.io"
    data_url = "ipfs://bafybeihfqxfckruepjhrkafaz6xg5a4sepx6ahhv4zds4b3hnfiyj35c5i"
    ds = xr.open_dataset(data_url, engine="zarr").load()

    # Convert StringDType vars to object dtype so NetCDF4 can save them
    # as variable-length strings instead of dropping them entirely
    str_vars = [v for v in ds.data_vars
                if ds[v].dtype.kind in ('O', 'U') or 'str' in str(ds[v].dtype).lower()]
    if str_vars:
        print(f"  Converting string vars to object dtype: {str_vars}")
        encoding_fixes = {}
        for v in str_vars:
            ds[v] = ds[v].astype(object)
            encoding_fixes[v] = {"dtype": "S200"}  # fixed-length bytes, 200 chars max

    ds.to_netcdf(LOCAL_PATH, encoding=encoding_fixes if str_vars else {})
    print(f"Downloaded and saved to: {LOCAL_PATH}")
    print(f"  Saved vars: {list(ds.data_vars)}")

print(f"Circles: {ds.sizes.get('circle', '?')}")
assert 'omega' in ds, "ERROR: 'omega' missing — check the dataset source."
assert 'p_mean' in ds, "ERROR: 'p_mean' missing — check the dataset source."
print("All required variables present. Ready to proceed.")

Loading from local cache: /g/data/k10/zr7147/raw/orcestra_level4.nc
Loaded successfully. Variables: ['circle_altitude', 'circle_id', 'circle_radius', 'div', 'div_sonde_relevance', 'div_std_error', 'flight_id', 'iwv', 'iwv_mean', 'launch_altitude', 'launch_lat', 'launch_lon', 'omega', 'omega_sonde_relevance', 'omega_std_error', 'p', 'p_dpdx', 'p_dpdx_std_error', 'p_dpdy', 'p_dpdy_std_error', 'p_mean', 'platform_id', 'q', 'q_dqdx', 'q_dqdx_std_error', 'q_dqdy', 'q_dqdy_std_error', 'q_mean', 'rh', 'rh_drhdx', 'rh_drhdx_std_error', 'rh_drhdy', 'rh_drhdy_std_error', 'rh_mean', 'sonde_id', 'sonde_qc', 'ta', 'ta_dtadx', 'ta_dtadx_std_error', 'ta_dtady', 'ta_dtady_std_error', 'ta_mean', 'theta', 'theta_dthetadx', 'theta_dthetadx_std_error', 'theta_dthetady', 'theta_dthetady_std_error', 'theta_mean', 'u', 'u_dudx', 'u_dudx_std_error', 'u_dudy', 'u_dudy_std_error', 'u_mean', 'v', 'v_dvdx', 'v_dvdx_std_error', 'v_dvdy', 'v_dvdy_std_error', 'v_mean', 'vaisala_serial_id', 'vor', 'vor_sonde_relevanc